# 01 - Ingest the tennis data
Run after 00_gcp_setup

Produces one compressed CSV file holding every season (header row once) in the bucket under raw/, and a manifest

## Install the pinned libraries

In [ ]:

# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v1"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "01_ingest_tennis"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Download from the pinned commit, combine, and upload to the bucket
The commit ID pins the exact snapshot of the mirror, so all users download the same files.
The per-season files are checked (same columns in the same order, as BigQuery matches columns by position) and combined into one gzip-compressed CSV instead of 17 small files.
The checksum and row count of every downloaded file are still recorded in the manifest, so the combined file can be traced back to its source files.
The combined file is written once: if it already exists with different content, the notebook stops (raw data is never overwritten).

In [ ]:
import gzip
import hashlib
import json
import requests
from datetime import datetime, timezone
from google.cloud import storage

COMMIT = "83733587353df8a41f2fd4f516147d5aa83f5a8d"   # pins the exact snapshot
assert len(COMMIT) == 40, "Use the full 40-character commit ID"
YEARS = range(2010, 2027)                             # 2010 through 2026

BASE_URL = f"https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/{COMMIT}/wta"
FOLDER = f"raw/tennis_sackmann_archive/{COMMIT[:8]}/wta"

bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
manifest = []          # one entry per downloaded source file
contents = []          # the downloaded files, in season order

for year in YEARS:
    name = f"wta_matches_{year}.csv"
    url = f"{BASE_URL}/{name}"

    # 1. Download
    response = requests.get(url, timeout=60)
    if response.status_code == 404:
        print(f"{name}: not found")
        continue
    response.raise_for_status()
    data = response.content

    # 2. Record what we got
    sha = hashlib.sha256(data).hexdigest()
    rows = data.count(b"\n") - 1
    header = data.decode("utf-8-sig").splitlines()[0]
    manifest.append({"file": name, "url": url, "rows": rows, "sha256": sha, "header": header})
    contents.append(data)
    print(f"{name}: downloaded | {rows:,} rows | {sha[:12]}")

# 3. Every file must have the same columns in the same order (BigQuery matches columns by position)
distinct = {e.pop("header") for e in manifest}
print(len(manifest), "files,", len(distinct), "distinct header row(s)")
assert len(distinct) == 1, "Headers differ between files: stop and investigate"
COLUMNS = next(iter(distinct)).split(",")
print(len(COLUMNS), "columns:", COLUMNS)

# 4. Combine: the first file whole, the others without their header row.
#    gzip with mtime=0, so the same input always gives the same bytes (and the same checksum).
combined = b""
for i, data in enumerate(contents):
    body = data if i == 0 else data.split(b"\n", 1)[1]
    if not body.endswith(b"\n"):
        body += b"\n"
    combined += body
packed = gzip.compress(combined, mtime=0)

years = [int(e["file"][12:16]) for e in manifest]
COMBINED = f"{FOLDER}/wta_matches_{years[0]}_{years[-1]}.csv.gz"
combined_entry = {"file": COMBINED, "bytes": len(packed), "sha256": sha256(packed),
                  "uncompressed_sha256": sha256(combined), "rows": sum(e["rows"] for e in manifest)}

# 5. Upload (written once, so raw data is never overwritten)
blob = bucket.blob(COMBINED)
if blob.exists():
    if sha256(blob.download_as_bytes()) != combined_entry["sha256"]:
        raise RuntimeError(f"{COMBINED} exists with different content: stop and investigate")
    status = "already stored, identical"
else:
    blob.upload_from_string(packed, content_type="application/gzip", if_generation_match=0)
    status = "uploaded"
print(f"{COMBINED}: {status} | {combined_entry['rows']:,} rows | "
      f"{len(combined) / 1e6:.1f} MB -> {len(packed) / 1e6:.1f} MB compressed")

# 6. Save the manifest (a timestamped record of this run) and add it to the index of data versions
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifest_path = f"manifests/wta_matches_{COMMIT[:8]}_{stamp}.json"
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"commit": COMMIT, "columns": COLUMNS, "files": [combined_entry],
                "source_files": manifest, **RUN_INFO}, indent=2),
    content_type="application/json",
)
print("Manifest saved:", manifest_path)
register(bucket, "raw/wta_matches", combined_entry["sha256"][:12], outputs=[combined_entry],
         parents=[f"github.com/Aneeshers/tennis-sackmann-archive@{COMMIT}"], manifest_path=manifest_path)

## Check the stored file is identical to the download
Per-season CSVs from earlier runs (`raw/tennis_sackmann_archive/83733587/wta/wta_matches_20XX.csv`) are no longer used and can be deleted from the bucket.

In [ ]:
stored = bucket.blob(COMBINED).download_as_bytes()
unpacked = gzip.decompress(stored)
checks = {
    "stored file checksum": sha256(stored) == combined_entry["sha256"],
    "content checksum after unpacking": sha256(unpacked) == combined_entry["uncompressed_sha256"],
    "rows after unpacking": unpacked.count(b"\n") - 1 == combined_entry["rows"],
}
for k, ok in checks.items():
    print("OK  " if ok else "FAIL", k)
assert all(checks.values()), "The stored file differs from the download"